In [10]:
import pandas as pd
import numpy as np
from scipy import stats

# ── Load data ─────────────────────────────────────────────────────────────────

df = pd.read_csv('birdsong.csv')
df['year']  = df['date'].str[:4].astype(int)
df['month'] = df['date'].str[5:7].astype(int)

# ── Define migration windows ──────────────────────────────────────────────────

spring_months = [3, 4, 5]    # Mar, Apr, May
autumn_months = [9, 10, 11]  # Sep, Oct, Nov

# ── Raw observation data (before any transformation) ──────────────
#
# One row per species-county-month observation. Year and month are parsed from
# the date string but the data is otherwise unchanged from the source file.

print("=" * 70)
print("Raw observation data (before transformation)")
print("=" * 70)
print(df[['common_name', 'date', 'year', 'month', 'season', 'county', 'bird_count']]
      .head(8)
      .to_string(index=False))
print(f"\nTotal rows: {len(df):,}  |  Unique species: {df['common_name'].nunique()}"
      f"  |  Years: {sorted(df['year'].unique())}")

# ── Monthly aggregation per species per year ──────────────────────────────────
#
# Sum bird counts across all counties for each species-year-month combination.
# This collapses the spatial dimension and produces one total count per species
# per calendar month per year — the unit needed to compute a centroid.

species_monthly = (
    df.groupby(['year', 'month', 'common_name'])['bird_count']
    .sum()
    .reset_index()
)

# ── After monthly aggregation ────────────────────────────────────
#
# One row per species-year-month. The number of rows drops from ~349,000 to
# one per unique (year, month, species) combination. Shown here for three
# selected species in spring 2021 to illustrate the structure.

print("\n" + "=" * 70)
print("After monthly aggregation per species per year")
print("=" * 70)
snap2 = species_monthly[
    species_monthly['common_name'].isin(['American Robin', 'Canada Goose', 'House Sparrow']) &
    (species_monthly['year'] == 2021) &
    (species_monthly['month'].isin(spring_months))
].sort_values(['common_name', 'month'])
print(snap2.to_string(index=False))
print(f"\nTotal rows after aggregation: {len(species_monthly):,}"
      f"  (down from {len(df):,} raw rows)")

# ── Compute per-species phenological centroids per year ───────────────────────
#
# Phenological centroid = weighted mean month of bird counts within the window:
#   centroid = Σ(month × bird_count) / Σ(bird_count)
#
# This gives a continuous measure of "when migration is centred" for each
# species × year combination — more sensitive than a single peak month.

records = []
for (year, species), grp in species_monthly.groupby(['year', 'common_name']):
    sp = grp[grp['month'].isin(spring_months)]
    au = grp[grp['month'].isin(autumn_months)]

    if sp['bird_count'].sum() > 0:
        sc = np.average(sp['month'], weights=sp['bird_count'])
        records.append({'year': year, 'species': species,
                        'window': 'spring', 'centroid': sc})
    if au['bird_count'].sum() > 0:
        ac = np.average(au['month'], weights=au['bird_count'])
        records.append({'year': year, 'species': species,
                        'window': 'autumn', 'centroid': ac})

rdf = pd.DataFrame(records)

# ── After phenological centroid computation ──────────────────────
#
# One centroid value per species per year per window. A centroid of 3.79 means
# activity is concentrated just after the start of March. Shown here for three
# selected species across all five years in the spring window.

print("\n" + "=" * 70)
print("After phenological centroid computation (spring window)")
print("=" * 70)
snap3 = rdf[
    rdf['species'].isin(['American Robin', 'Canada Goose', 'House Sparrow']) &
    (rdf['window'] == 'spring')
].sort_values(['species', 'year'])
print(snap3.to_string(index=False))
print(f"\nTotal centroid records (all windows): {len(rdf):,}"
      f"  ({rdf['window'].value_counts().to_dict()})")

# ── Compute per-species shifts: late years minus early years ──────────────────
#
# For each species, average the centroid over the early period (2021-22) and
# the late period (2024-25), then subtract. The resulting shift distribution
# is the direct input to the Wilcoxon signed-rank test.

early_years = [2021, 2022]
late_years  = [2024, 2025]

# ── After shift computation (Wilcoxon test input) ────────────────
#
# One shift value per species. Negative = species arrived earlier in the late
# period; positive = later; zero = no change. Shown here for six representative
# species in the spring window.

print("\n" + "=" * 70)
print("After shift computation (direct input to Wilcoxon test)")
print("=" * 70)
print("  Spring window  (shift = late_centroid - early_centroid)")
print(f"  {'Species':<35}  {'Early centroid':>14}  {'Late centroid':>13}  {'Shift':>7}")
print(f"  {'-' * 74}")

sample_species = ['American Robin', 'Barn Swallow', 'Canada Goose',
                  'Cliff Swallow', 'House Sparrow', 'Ruby-throated Hummingbird']
wdf_sp = rdf[rdf['window'] == 'spring']
early_sp = wdf_sp[wdf_sp['year'].isin(early_years)].groupby('species')['centroid'].mean()
late_sp  = wdf_sp[wdf_sp['year'].isin(late_years)].groupby('species')['centroid'].mean()
for sp in sample_species:
    if sp in early_sp.index and sp in late_sp.index:
        e, l = early_sp[sp], late_sp[sp]
        s = l - e
        direction = 'earlier' if s < -0.01 else 'later' if s > 0.01 else 'unchanged'
        print(f"  {sp:<35}  {e:>14.4f}  {l:>13.4f}  {s:>+7.4f}  ({direction})")

# ── Run hypothesis tests ──────────────────────────────────────────────────────

for window in ['spring', 'autumn']:
    wdf   = rdf[rdf['window'] == window]
    early = wdf[wdf['year'].isin(early_years)].groupby('species')['centroid'].mean()
    late  = wdf[wdf['year'].isin(late_years)].groupby('species')['centroid'].mean()

    common = early.index.intersection(late.index)
    shifts = (late[common] - early[common]).values   # negative = earlier arrival

    n           = len(shifts)
    mean_shift  = shifts.mean()
    std_shift   = shifts.std()
    pct_earlier = (shifts < 0).mean() * 100
    pct_later   = (shifts > 0).mean() * 100
    cohens_d    = mean_shift / std_shift

    # Non-parametric test (no normality assumption)
    stat_w, p_wilcox = stats.wilcoxon(shifts)

    # Parametric cross-check
    t_stat, p_t = stats.ttest_1samp(shifts, 0)

    print(f"\n{'='*60}")
    print(f"{window.capitalize()} migration centroid shift (late - early years)")
    print(f"{'='*60}")
    print(f"  Species with data in both periods : {n}")
    print(f"  Mean shift                        : {mean_shift:.4f} months"
          f"  ({mean_shift * 30.4:.1f} days)")
    print(f"  Std of shifts                     : {std_shift:.4f}")
    print(f"  % shifted earlier (negative)      : {pct_earlier:.1f}%")
    print(f"  % shifted later   (positive)      : {pct_later:.1f}%")
    print(f"  Wilcoxon signed-rank  stat={stat_w:.1f}  p={p_wilcox:.4f}")
    print(f"  One-sample t-test     t={t_stat:.3f}      p={p_t:.4f}")
    print(f"  Cohen's d             : {cohens_d:.4f}")

# ── Year-by-year mean centroid across all species (for trend inspection) ──────

print("\n\nYear-by-year mean phenological centroid")
print("-" * 45)
for window in ['spring', 'autumn']:
    wdf = rdf[rdf['window'] == window]
    print(f"\n{window.capitalize()}:")
    for year in sorted(rdf['year'].unique()):
        vals = wdf[wdf['year'] == year]['centroid'].values
        print(f"  {year}  mean={vals.mean():.4f}  std={vals.std():.4f}"
              f"  n={len(vals)}")

Raw observation data (before transformation)
                   common_name    date  year  month season   county  bird_count
            Band-tailed Pigeon 2021-01  2021      1 Winter Garfield         1.0
                  Pine Warbler 2021-01  2021      1 Winter  Boulder         1.0
           White-winged Scoter 2021-01  2021      1 Winter  Larimer         1.0
                Brown Thrasher 2021-01  2021      1 Winter  Boulder         1.0
              Bonaparte's Gull 2021-01  2021      1 Winter   Pueblo         2.0
American Three-toed Woodpecker 2021-01  2021      1 Winter  Larimer         1.0
            Greater Roadrunner 2021-01  2021      1 Winter   Pueblo         1.0
 Graylag x Swan Goose (hybrid) 2021-01  2021      1 Winter   Denver         3.0

Total rows: 349,430  |  Unique species: 567  |  Years: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

After monthly aggregation per species per year
 year  month    common_name  bird_count
 2021     

In [4]:
import pandas as pd
import numpy as np
from scipy import stats

# ── Load data ─────────────────────────────────────────────────────────────────

df = pd.read_csv('birdsong.csv')
df['year']  = df['date'].str[:4].astype(int)
df['month'] = df['date'].str[5:7].astype(int)

# ── Compute per-species full-year phenological centroids ──────────────────────
#
# Phenological centroid = weighted mean month across all 12 months:
#   centroid = Σ(month × bird_count) / Σ(bird_count)
#
# Unlike the seasonal windows, this captures where the entire annual
# "centre of mass" of each species' presence sits — a shift toward
# later months means more activity is occurring later in the year.

species_monthly = (
    df.groupby(['year', 'month', 'common_name'])['bird_count']
    .sum()
    .reset_index()
)

records = []
for (year, species), grp in species_monthly.groupby(['year', 'common_name']):
    total = grp['bird_count'].sum()
    if total > 0:
        centroid = np.average(grp['month'], weights=grp['bird_count'])
        records.append({'year': year, 'species': species, 'centroid': centroid})

rdf = pd.DataFrame(records)
print(f"Total centroid records: {len(rdf)}")

# ── Compute per-species shifts: late years minus early years ──────────────────

early_years = [2021, 2022]
late_years  = [2024, 2025]

early = rdf[rdf['year'].isin(early_years)].groupby('species')['centroid'].mean()
late  = rdf[rdf['year'].isin(late_years)].groupby('species')['centroid'].mean()

common = early.index.intersection(late.index)
shifts = (late[common] - early[common]).values   # positive = later in year

n           = len(shifts)
mean_shift  = shifts.mean()
std_shift   = shifts.std()
pct_earlier = (shifts < 0).mean() * 100
pct_later   = (shifts > 0).mean() * 100
cohens_d    = mean_shift / std_shift

# Non-parametric test (no normality assumption)
stat_w, p_wilcox = stats.wilcoxon(shifts)

# Parametric cross-check
t_stat, p_t = stats.ttest_1samp(shifts, 0)

print(f"\n{'='*60}")
print(f"Full-year phenological centroid shift (late - early years)")
print(f"{'='*60}")
print(f"  Species with data in both periods : {n}")
print(f"  Mean shift                        : {mean_shift:.4f} months"
      f" ({mean_shift * 30.4:.1f} days)")
print(f"  Std of shifts                     : {std_shift:.4f}")
print(f"  % shifted earlier (negative)      : {pct_earlier:.1f}%")
print(f"  % shifted later   (positive)      : {pct_later:.1f}%")
print(f"  Wilcoxon signed-rank  stat={stat_w:.1f}  p={p_wilcox:.4f}")
print(f"  One-sample t-test     t={t_stat:.3f}      p={p_t:.4f}")
print(f"  Cohen's d             : {cohens_d:.4f}")

# ── Year-by-year mean centroid across all species (for trend inspection) ──────

print("\n\nYear-by-year mean full-year centroid")
print("-" * 45)
for year in sorted(rdf['year'].unique()):
    vals = rdf[rdf['year'] == year]['centroid'].values
    print(f"  {year}  mean={vals.mean():.4f}  std={vals.std():.4f}"
          f"  n={len(vals)}")

# ── Summary comparison across all three tests ─────────────────────────────────
#
# For reference, results from the seasonal window tests run previously:
#
#   Spring (Mar-May):  mean shift = -0.0859 mo (-2.6 days)
#                      Wilcoxon p < 0.0001, t-test p < 0.0001, d = -0.28
#                      67.8% of species shifted EARLIER  --> SIGNIFICANT
#
#   Autumn (Sep-Nov):  mean shift = -0.0074 mo (-0.2 days)
#                      Wilcoxon p = 0.209,   t-test p = 0.641,  d = -0.02
#                      40.6% of species shifted earlier  --> NOT SIGNIFICANT
#
#   Full year:         mean shift = +0.1444 mo (+4.4 days)
#                      Wilcoxon p < 0.0001,  t-test p = 0.035,  d = +0.10
#                      66.7% of species shifted LATER    --> SIGNIFICANT
#
# Interpretation: spring arrivals are advancing (earlier), while the full-year
# centre of mass is shifting later — suggesting increased late-season activity
# (summer residency, winter presence) is pulling the annual centroid forward
# even as birds arrive earlier in spring.

Total centroid records: 2429

Full-year phenological centroid shift (late - early years)
  Species with data in both periods : 475
  Mean shift                        : 0.1444 months (4.4 days)
  Std of shifts                     : 1.4869
  % shifted earlier (negative)      : 33.1%
  % shifted later   (positive)      : 66.7%
  Wilcoxon signed-rank  stat=39739.5  p=0.0000
  One-sample t-test     t=2.114      p=0.0350
  Cohen's d             : 0.0971


Year-by-year mean full-year centroid
---------------------------------------------
  2021  mean=6.4962  std=1.7293  n=478
  2022  mean=6.5519  std=1.5357  n=481
  2023  mean=6.5696  std=1.6660  n=488
  2024  mean=6.5835  std=1.6432  n=491
  2025  mean=6.7642  std=1.7058  n=491


In [5]:
import pandas as pd
import numpy as np
from scipy import stats

# ── Load data ─────────────────────────────────────────────────────────────────

df = pd.read_csv('birdsong.csv')
df['year']  = df['date'].str[:4].astype(int)
df['month'] = df['date'].str[5:7].astype(int)

# ── Season definitions ────────────────────────────────────────────────────────

season_months = {
    'Winter': [12, 1, 2],
    'Spring': [3, 4, 5],
    'Summer': [6, 7, 8],
    'Autumn': [9, 10, 11],
}

# ── Winter centroid helper ────────────────────────────────────────────────────
#
# December (month 12) is numerically far from January (1) and February (2),
# so a naive weighted mean would be distorted. We remap the winter months
# onto a continuous 0–2 scale before computing the centroid:
#   Dec → 0,  Jan → 1,  Feb → 2
# A centroid of 0.9 therefore means "slightly after mid-December".

def winter_centroid(grp):
    months = grp['month'].map({12: 0, 1: 1, 2: 2})
    return np.average(months, weights=grp['bird_count'])

# ── Compute per-species seasonal phenological centroids ───────────────────────

species_monthly = (
    df.groupby(['year', 'month', 'common_name'])['bird_count']
    .sum()
    .reset_index()
)

records = []
for (year, species), grp in species_monthly.groupby(['year', 'common_name']):
    for season, months in season_months.items():
        sub = grp[grp['month'].isin(months)]
        if sub['bird_count'].sum() > 0:
            if season == 'Winter':
                c = winter_centroid(sub)
            else:
                c = np.average(sub['month'], weights=sub['bird_count'])
            records.append({'year': year, 'species': species,
                            'season': season, 'centroid': c})

rdf = pd.DataFrame(records)
print(f"Total centroid records: {len(rdf)}")

# ── Compute per-species shifts and run hypothesis tests ───────────────────────

early_years = [2021, 2022]
late_years  = [2024, 2025]

for season in ['Winter', 'Spring', 'Summer', 'Autumn']:
    wdf   = rdf[rdf['season'] == season]
    early = wdf[wdf['year'].isin(early_years)].groupby('species')['centroid'].mean()
    late  = wdf[wdf['year'].isin(late_years)].groupby('species')['centroid'].mean()

    common = early.index.intersection(late.index)
    shifts = (late[common] - early[common]).values   # negative = earlier

    n           = len(shifts)
    mean_shift  = shifts.mean()
    std_shift   = shifts.std()
    pct_earlier = (shifts < 0).mean() * 100
    pct_later   = (shifts > 0).mean() * 100
    cohens_d    = mean_shift / std_shift

    # Non-parametric test (no normality assumption)
    stat_w, p_wilcox = stats.wilcoxon(shifts)

    # Parametric cross-check
    t_stat, p_t = stats.ttest_1samp(shifts, 0)

    print(f"\n{'='*60}")
    print(f"{season}")
    print(f"{'='*60}")
    print(f"  Species with data in both periods : {n}")
    print(f"  Mean shift                        : {mean_shift:.4f} months"
          f" ({mean_shift * 30.4:.1f} days)")
    print(f"  Std of shifts                     : {std_shift:.4f}")
    print(f"  % shifted earlier (negative)      : {pct_earlier:.1f}%")
    print(f"  % shifted later   (positive)      : {pct_later:.1f}%")
    print(f"  Wilcoxon signed-rank  stat={stat_w:.1f}  p={p_wilcox:.4f}")
    print(f"  One-sample t-test     t={t_stat:.3f}      p={p_t:.4f}")
    print(f"  Cohen's d             : {cohens_d:.4f}")

# ── Year-by-year mean centroids (for trend inspection) ───────────────────────

print("\n\nYear-by-year mean phenological centroid")
print("-" * 45)
for season in ['Winter', 'Spring', 'Summer', 'Autumn']:
    wdf = rdf[rdf['season'] == season]
    print(f"\n{season}:")
    for year in sorted(rdf['year'].unique()):
        vals = wdf[wdf['year'] == year]['centroid'].values
        print(f"  {year}  mean={vals.mean():.4f}  std={vals.std():.4f}"
              f"  n={len(vals)}")

# ── Results summary ───────────────────────────────────────────────────────────
#
#   Winter:  mean shift = -0.0162 mo (-0.5 days)
#            Wilcoxon p = 0.335,   t-test p = 0.514,  d = -0.04
#            50.0% earlier / 47.9% later  --> NOT SIGNIFICANT
#
#   Spring:  mean shift = -0.0859 mo (-2.6 days)
#            Wilcoxon p < 0.0001,  t-test p < 0.0001, d = -0.28
#            67.8% earlier            --> SIGNIFICANT (arriving earlier)
#
#   Summer:  mean shift = +0.0600 mo (+1.8 days)
#            Wilcoxon p < 0.0001,  t-test p = 0.001,  d = +0.17
#            60.1% later              --> SIGNIFICANT (shifting later)
#
#   Autumn:  mean shift = -0.0074 mo (-0.2 days)
#            Wilcoxon p = 0.209,   t-test p = 0.641,  d = -0.02
#            40.6% earlier / 51.3% later  --> NOT SIGNIFICANT
#
# Interpretation: spring arrivals are advancing (earlier) while summer
# activity is elongating toward later in August — consistent with a
# lengthening of the active season. Autumn departure and winter timing
# show no statistically significant change.

Total centroid records: 7472

Winter
  Species with data in both periods : 280
  Mean shift                        : -0.0162 months (-0.5 days)
  Std of shifts                     : 0.4145
  % shifted earlier (negative)      : 50.0%
  % shifted later   (positive)      : 47.9%
  Wilcoxon signed-rank  stat=17571.5  p=0.3349
  One-sample t-test     t=-0.654      p=0.5139
  Cohen's d             : -0.0391

Spring
  Species with data in both periods : 428
  Mean shift                        : -0.0859 months (-2.6 days)
  Std of shifts                     : 0.3089
  % shifted earlier (negative)      : 67.8%
  % shifted later   (positive)      : 24.1%
  Wilcoxon signed-rank  stat=17966.5  p=0.0000
  One-sample t-test     t=-5.748      p=0.0000
  Cohen's d             : -0.2782

Summer
  Species with data in both periods : 368
  Mean shift                        : 0.0600 months (1.8 days)
  Std of shifts                     : 0.3572
  % shifted earlier (negative)      : 36.4%
  % shifted later

In [11]:
import pandas as pd
import numpy as np
from scipy import stats

# ── Load data ─────────────────────────────────────────────────────────────────

df = pd.read_csv('birdsong.csv')
df['year']  = df['date'].str[:4].astype(int)
df['month'] = df['date'].str[5:7].astype(int)

month_names = {1:'Jan', 2:'Feb',  3:'Mar',  4:'Apr',  5:'May',  6:'Jun',
               7:'Jul', 8:'Aug',  9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}

# ── SNAPSHOT 1: Raw observation data (before any transformation) ──────────────
#
# One row per species-county-month observation. Bird counts are repeated across
# many counties for the same species-month combination.

print("=" * 70)
print("SNAPSHOT 1: Raw observation data (before transformation)")
print("=" * 70)
print(df[['common_name', 'date', 'year', 'month', 'county', 'bird_count']]
      .head(8)
      .to_string(index=False))
print(f"\nTotal rows: {len(df):,}  |  Unique species: {df['common_name'].nunique()}"
      f"  |  Years: {sorted(df['year'].unique())}")

# ── Monthly aggregation per species per year ──────────────────────────────────
#
# Sum bird counts across all counties for each species-year-month combination.
# This collapses the spatial dimension so each species has one count per
# calendar month per year.

species_monthly = (
    df.groupby(['year', 'month', 'common_name'])['bird_count']
    .sum()
    .reset_index()
)

# ── SNAPSHOT 2: After monthly aggregation per species per year ────────────────
#
# One row per (year, month, species). Shown here for three species in May to
# illustrate the count structure that feeds the Wilcoxon test.

print("\n" + "=" * 70)
print("SNAPSHOT 2: After monthly aggregation per species per year")
print("=" * 70)
snap2 = species_monthly[
    species_monthly['common_name'].isin(['American Robin', 'Canada Goose', 'House Sparrow']) &
    (species_monthly['month'] == 5)
].sort_values(['common_name', 'year'])
print(snap2.to_string(index=False))
print(f"\nTotal rows after aggregation: {len(species_monthly):,}"
      f"  (down from {len(df):,} raw rows)")

# ── Compute per-species mean counts for early and late periods ────────────────
#
# For each month, average each species' count over the early years (2021-22)
# and the late years (2024-25), then subtract to get a shift value.
# The shift distribution (one value per species) is the direct input to the
# Wilcoxon signed-rank test.

early_years = [2021, 2022]
late_years  = [2024, 2025]

# ── SNAPSHOT 3: After early/late mean computation (May example) ──────────────
#
# Shows the per-species mean count in each period for May, before differencing.
# Positive diff = more birds in late years; negative = fewer.

print("\n" + "=" * 70)
print("SNAPSHOT 3: Per-species mean counts by period (May, selected species)")
print("=" * 70)
may_df    = species_monthly[species_monthly['month'] == 5]
early_may = may_df[may_df['year'].isin(early_years)].groupby('common_name')['bird_count'].mean()
late_may  = may_df[may_df['year'].isin(late_years)].groupby('common_name')['bird_count'].mean()
common_may = early_may.index.intersection(late_may.index)

sample_sp = ['American Robin', 'Barn Swallow', 'Canada Goose',
             'House Sparrow', 'White-throated Warbler', 'Wilson\'s Warbler']
snap3_rows = []
for sp in sample_sp:
    if sp in common_may:
        e, l = early_may[sp], late_may[sp]
        snap3_rows.append({'species': sp, 'early_mean': round(e, 2),
                           'late_mean': round(l, 2), 'diff (late-early)': round(l - e, 2)})
snap3_df = pd.DataFrame(snap3_rows)
print(snap3_df.to_string(index=False))
print(f"\nSpecies with data in both periods (May): {len(common_may):,}")

# ── SNAPSHOT 4: After shift computation (Wilcoxon test input, May) ────────────
#
# The full distribution of per-species diffs for May — one value per species.
# This is the array passed directly to stats.wilcoxon(). A negative value
# means that species had lower counts in the late period (fewer birds in May).

print("\n" + "=" * 70)
print("SNAPSHOT 4: Shift distribution summary (May — direct Wilcoxon input)")
print("=" * 70)
diffs_may = (late_may[common_may] - early_may[common_may]).values
print(f"  n species  : {len(diffs_may):,}")
print(f"  Mean diff  : {diffs_may.mean():+.2f}  birds")
print(f"  Median diff: {np.median(diffs_may):+.2f}  birds")
print(f"  % decreased (negative): {(diffs_may < 0).mean()*100:.1f}%")
print(f"  % increased (positive): {(diffs_may > 0).mean()*100:.1f}%")
print(f"  Min / Max  : {diffs_may.min():.1f} / {diffs_may.max():.1f}")

# ── Wilcoxon signed-rank test per month ───────────────────────────────────────
#
# For each calendar month, build two groups — early years (2021-22) and late
# years (2024-25) — as per-species mean counts. The difference (late - early)
# forms the shift distribution. The Wilcoxon signed-rank test asks whether the
# median shift is significantly different from zero.
#
# H0: the median shift in per-species monthly count = 0
# H1: the median shift != 0

print("\n" + "=" * 70)
print("WILCOXON SIGNED-RANK TEST: monthly bird count shift (early vs late years)")
print("=" * 70)
print(f"\n  {'Month':<5}  {'n':>4}  {'Mean diff':>10}  {'% down':>7}  "
      f"{'Wilcoxon p':>11}  {'t-test p':>9}  {'Cohen d':>8}  Sig")
print(f"  {'-' * 72}")

results = {}
for month in range(1, 13):
    mdf   = species_monthly[species_monthly['month'] == month]
    early = mdf[mdf['year'].isin(early_years)].groupby('common_name')['bird_count'].mean()
    late  = mdf[mdf['year'].isin(late_years)].groupby('common_name')['bird_count'].mean()

    common = early.index.intersection(late.index)
    diffs  = (late[common] - early[common]).values

    n            = len(diffs)
    mean_diff    = diffs.mean()
    pct_decrease = (diffs < 0).mean() * 100
    cohens_d     = mean_diff / diffs.std() if diffs.std() > 0 else 0

    stat_w, p_w = stats.wilcoxon(diffs)
    t_stat, p_t = stats.ttest_1samp(diffs, 0)

    sig = '***' if p_w < 0.001 else '**' if p_w < 0.01 \
          else '*' if p_w < 0.05 else 'n.s.'

    results[month] = dict(name=month_names[month], n=n, mean_diff=mean_diff,
                          pct_decrease=pct_decrease, cohens_d=cohens_d,
                          p_wilcox=p_w, p_t=p_t, sig=sig)

    print(f"  {month_names[month]:<5}  {n:>4}  {mean_diff:>+10.2f}  "
          f"{pct_decrease:>6.1f}%  {p_w:>11.4f}  {p_t:>9.4f}  "
          f"{cohens_d:>+8.3f}  {sig}")

# ── Year-by-year mean counts per month (for trend inspection) ─────────────────

print("\n\nYear-by-year mean bird count per species per month")
print("-" * 55)
print(f"  {'Month':<5}  {'2021':>8}  {'2022':>8}  {'2023':>8}  {'2024':>8}  {'2025':>8}")
print(f"  {'-' * 52}")
for month in range(1, 13):
    mdf  = species_monthly[species_monthly['month'] == month]
    vals = [mdf[mdf['year'] == y]['bird_count'].mean() for y in [2021,2022,2023,2024,2025]]
    print(f"  {month_names[month]:<5}  " + "  ".join(f"{v:>8.1f}" for v in vals))

# ── Summary of significant months ────────────────────────────────────────────

print("\n\nSignificant months (Wilcoxon p < 0.05):")
print("-" * 45)
for month, r in results.items():
    if r['p_wilcox'] < 0.05:
        print(f"  {r['name']}  p={r['p_wilcox']:.4f} {r['sig']}"
              f"  mean_diff={r['mean_diff']:+.2f}"
              f"  {r['pct_decrease']:.1f}% of species decreased"
              f"  Cohen's d={r['cohens_d']:+.3f}")

SNAPSHOT 1: Raw observation data (before transformation)
                   common_name    date  year  month   county  bird_count
            Band-tailed Pigeon 2021-01  2021      1 Garfield         1.0
                  Pine Warbler 2021-01  2021      1  Boulder         1.0
           White-winged Scoter 2021-01  2021      1  Larimer         1.0
                Brown Thrasher 2021-01  2021      1  Boulder         1.0
              Bonaparte's Gull 2021-01  2021      1   Pueblo         2.0
American Three-toed Woodpecker 2021-01  2021      1  Larimer         1.0
            Greater Roadrunner 2021-01  2021      1   Pueblo         1.0
 Graylag x Swan Goose (hybrid) 2021-01  2021      1   Denver         3.0

Total rows: 349,430  |  Unique species: 567  |  Years: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

SNAPSHOT 2: After monthly aggregation per species per year
 year  month    common_name  bird_count
 2021      5 American Robin        80.0
 2022    

In [7]:
import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations

# ── Load data ─────────────────────────────────────────────────────────────────

df = pd.read_csv('birdsong.csv')
df['year']  = df['date'].str[:4].astype(int)
df['month'] = df['date'].str[5:7].astype(int)

month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

# ── Aggregate to per-species mean count per year × month ─────────────────────

species_monthly = (
    df.groupby(['year', 'month', 'common_name'])['bird_count']
    .sum()
    .reset_index()
)

# ── Kruskal-Wallis H-test across all 5 years, run once per month ──────────────
#
# For each month, we build 5 groups — one per year — where each group is an
# array of per-species mean bird counts for that year × month combination.
# KW tests H0: all 5 year-groups are drawn from the same distribution.
# Unlike the earlier Wilcoxon (which collapsed years into early/late bins),
# KW treats every year independently, making it harder to reach significance
# but more informative about *which* years differ.

print("=== Kruskal-Wallis H-test: per-month bird counts across all 5 years ===\n")
print(f"{'Month':<5} {'H':>8} {'p':>10} {'Sig':>5}  "
      f"Median counts (2021 → 2025)")
print("-" * 75)

results = {}
for month in range(1, 13):
    mdf = species_monthly[species_monthly['month'] == month]

    groups  = []
    medians = []
    for year in [2021, 2022, 2023, 2024, 2025]:
        vals = (mdf[mdf['year'] == year]
                .groupby('common_name')['bird_count']
                .mean()
                .values)
        groups.append(vals)
        medians.append(np.median(vals))

    h_stat, p_val = stats.kruskal(*groups)

    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 \
          else '*' if p_val < 0.05 else 'n.s.'
    med_str = '  '.join(f'{m:.1f}' for m in medians)

    print(f"{month_names[month]:<5} {h_stat:>8.3f} {p_val:>10.4f} {sig:>5}  "
          f"{med_str}")

    results[month] = dict(name=month_names[month], h=h_stat, p=p_val,
                          sig=sig, medians=medians, groups=groups)

# ── Post-hoc pairwise Mann-Whitney U for significant months ───────────────────
#
# KW only tells us *some* year differs — not which ones. For every month that
# reaches significance (p < 0.05) we run all 10 pairwise Mann-Whitney U tests
# between the 5 years. No Bonferroni correction is applied here; apply
# alpha = 0.005 manually if you want a conservative threshold for 10 pairs.

print("\n\n=== Post-hoc pairwise Mann-Whitney U (significant months only) ===")
years = [2021, 2022, 2023, 2024, 2025]

for month, r in results.items():
    if r['p'] >= 0.05:
        continue
    print(f"\n{r['name']}  (H = {r['h']:.3f},  p = {r['p']:.4f})")
    print(f"  {'Pair':<18} {'U':>8}  {'p':>10}  Sig")
    print(f"  {'-'*44}")
    for i, j in combinations(range(5), 2):
        u, p = stats.mannwhitneyu(r['groups'][i], r['groups'][j],
                                  alternative='two-sided')
        sig = '***' if p < 0.001 else '**' if p < 0.01 \
              else '*' if p < 0.05 else 'n.s.'
        print(f"  {years[i]} vs {years[j]:<10} {u:>8.0f}  {p:>10.4f}  {sig}")

# ── Key findings ──────────────────────────────────────────────────────────────
#
# Significant months (p < 0.05):
#   May  H = 155.518, p < 0.001  — post-hoc: ONLY pairs involving 2025 differ
#   Jun  H =  19.742, p = 0.001  — post-hoc: ONLY pairs involving 2025 differ
#
# All other months: not significant (p > 0.10)
#
# Comparison with prior Wilcoxon signed-rank test (early 2021-22 vs late 2024-25):
#   April was significant under Wilcoxon (*** p < 0.001) but NOT here (p = 0.147).
#   Reason: Wilcoxon collapsed years into two bins, amplifying a modest
#   directional trend. KW sees 2021-2024 as indistinguishable for April —
#   no single year is a dramatic outlier, so the omnibus test doesn't flag it.
#
# Interpretation: the May and June signals are driven entirely by an abrupt
# 2025 drop (May median: 38-39 in 2021-2024 → 14 in 2025). This is more
# consistent with a 2025 data anomaly (incomplete records?) than a gradual
# multi-year migration trend. Verify 2025 data completeness before drawing
# ecological conclusions.

=== Kruskal-Wallis H-test: per-month bird counts across all 5 years ===

Month        H          p   Sig  Median counts (2021 → 2025)
---------------------------------------------------------------------------
Jan      1.005     0.9090  n.s.  37.0  36.0  34.0  35.0  35.0
Feb      0.888     0.9263  n.s.  34.0  33.0  30.0  34.0  31.5
Mar      0.499     0.9736  n.s.  34.0  35.5  33.0  36.5  36.0
Apr      6.790     0.1474  n.s.  36.0  33.0  35.0  34.0  30.0
May    155.518     0.0000   ***  38.0  39.0  39.0  38.0  14.0
Jun     19.742     0.0006   ***  42.0  42.0  42.0  44.0  31.0
Jul      2.778     0.5957  n.s.  49.0  49.0  49.5  47.0  44.0
Aug      1.960     0.7430  n.s.  44.0  48.0  46.0  44.0  42.0
Sep      0.662     0.9559  n.s.  36.0  37.0  39.0  36.5  36.0
Oct      1.084     0.8969  n.s.  24.5  24.5  30.5  30.0  25.0
Nov      1.463     0.8332  n.s.  28.0  25.0  30.0  30.0  27.0
Dec      2.113     0.7150  n.s.  33.0  32.5  31.0  35.5  30.0


=== Post-hoc pairwise Mann-Whitney U (signif

In [8]:
import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations

# ── Load data ─────────────────────────────────────────────────────────────────

df = pd.read_csv('birdsong.csv')
df['year']  = df['date'].str[:4].astype(int)
df['month'] = df['date'].str[5:7].astype(int)

season_months = {
    'Winter': [12, 1, 2],
    'Spring': [3, 4, 5],
    'Summer': [6, 7, 8],
    'Autumn': [9, 10, 11],
}

# ── Aggregate to per-species seasonal totals per year ─────────────────────────
#
# For each species × year combination, sum bird counts across the three
# months that make up each season. This gives one value per species per
# season per year — the unit on which the KW test operates.

species_monthly = (
    df.groupby(['year', 'month', 'common_name'])['bird_count']
    .sum()
    .reset_index()
)

records = []
for (year, species), grp in species_monthly.groupby(['year', 'common_name']):
    for season, months in season_months.items():
        sub = grp[grp['month'].isin(months)]
        total = sub['bird_count'].sum()
        if total > 0:
            records.append({'year': year, 'species': species,
                            'season': season, 'bird_count': total})

sdf = pd.DataFrame(records)
print(f"Total seasonal records: {len(sdf)}")

# ── Kruskal-Wallis H-test across all 5 years, run once per season ─────────────
#
# For each season, build 5 groups — one per year — where each group is an
# array of per-species seasonal bird counts. KW tests H0: all 5 year-groups
# are drawn from the same distribution.

years = [2021, 2022, 2023, 2024, 2025]

print("\n=== Kruskal-Wallis H-test: seasonal bird counts across all 5 years ===\n")
print(f"{'Season':<8} {'H':>8} {'p':>10} {'Sig':>5}  "
      f"Median counts (2021 → 2025)")
print("-" * 75)

results = {}
for season in ['Winter', 'Spring', 'Summer', 'Autumn']:
    wdf = sdf[sdf['season'] == season]

    groups  = []
    medians = []
    for year in years:
        vals = (wdf[wdf['year'] == year]
                .groupby('species')['bird_count']
                .mean()
                .values)
        groups.append(vals)
        medians.append(np.median(vals))

    h_stat, p_val = stats.kruskal(*groups)
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 \
          else '*' if p_val < 0.05 else 'n.s.'
    med_str = '  '.join(f'{m:.1f}' for m in medians)

    print(f"{season:<8} {h_stat:>8.3f} {p_val:>10.4f} {sig:>5}  {med_str}")

    results[season] = dict(h=h_stat, p=p_val, sig=sig,
                           medians=medians, groups=groups)

# ── Post-hoc pairwise Mann-Whitney U for significant seasons ──────────────────
#
# KW only flags that *some* year differs — post-hoc pairs identify which.
# All 10 pairwise combinations of 5 years are tested with Mann-Whitney U.

print("\n\n=== Post-hoc pairwise Mann-Whitney U (significant seasons only) ===")

for season, r in results.items():
    if r['p'] >= 0.05:
        continue
    print(f"\n{season}  (H = {r['h']:.3f},  p = {r['p']:.4f})")
    print(f"  {'Pair':<18} {'U':>8}  {'p':>10}  Sig")
    print(f"  {'-'*44}")
    for i, j in combinations(range(5), 2):
        u, p = stats.mannwhitneyu(r['groups'][i], r['groups'][j],
                                  alternative='two-sided')
        sig = '***' if p < 0.001 else '**' if p < 0.01 \
              else '*' if p < 0.05 else 'n.s.'
        print(f"  {years[i]} vs {years[j]:<10} {u:>8.0f}  {p:>10.4f}  {sig}")

# ── Key findings ──────────────────────────────────────────────────────────────
#
# Significant seasons (p < 0.05):
#   Spring  H = 14.084, p = 0.007  — post-hoc: ONLY pairs involving 2025
#           differ; 2021–2024 are indistinguishable from each other.
#           Spring median drops from 67–72 (2021–2024) to 50 in 2025.
#
# Not significant:
#   Winter  H = 1.623,  p = 0.805
#   Summer  H = 4.596,  p = 0.331
#   Autumn  H = 0.933,  p = 0.920
#
# Consistent with the monthly KW test: the signal is isolated to Spring
# and driven entirely by 2025. Verify 2025 data completeness (especially
# for May) before treating this as a confirmed biological trend.

Total seasonal records: 7472

=== Kruskal-Wallis H-test: seasonal bird counts across all 5 years ===

Season          H          p   Sig  Median counts (2021 → 2025)
---------------------------------------------------------------------------
Winter      1.623     0.8047  n.s.  83.5  80.5  73.0  84.0  67.5
Spring     14.084     0.0070    **  69.0  69.0  72.0  67.0  50.0
Summer      4.596     0.3313  n.s.  120.0  128.5  126.5  131.0  112.0
Autumn      0.933     0.9199  n.s.  56.0  58.5  66.0  65.0  55.0


=== Post-hoc pairwise Mann-Whitney U (significant seasons only) ===

Spring  (H = 14.084,  p = 0.0070)
  Pair                      U           p  Sig
  --------------------------------------------
  2021 vs 2022          92404      0.8630  n.s.
  2021 vs 2023          94196      0.6238  n.s.
  2021 vs 2024          95130      0.4551  n.s.
  2021 vs 2025          98192      0.0010  ***
  2022 vs 2023          97038      0.7208  n.s.
  2022 vs 2024          98030      0.5335  n.s.
  2022 